# Albanian Text Summarization with mT5 and LoRA

End-to-end notebook for training, evaluating, and exporting a LoRA adapter for `google/mt5-small`. Run the cells in order from the project root. A CUDA GPU is strongly recommended.

## 1. Environment setup

Install only the packages that are not guaranteed to be available in a Kaggle or local notebook environment. PyTorch is intentionally not reinstalled so the CUDA build supplied by the environment is preserved.

In [ ]:
%pip install -q transformers datasets peft evaluate accelerate sentencepiece rouge_score

In [ ]:
from pathlib import Path

import evaluate
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

SEED = 42
MODEL_NAME = "google/mt5-small"
MAX_SOURCE_LENGTH = 640
MAX_TARGET_LENGTH = 96

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

IS_KAGGLE = Path("/kaggle/working").exists()
TRAINER_DIR = Path("/kaggle/working/mt5-training") if IS_KAGGLE else PROJECT_ROOT / "outputs/mt5-training"
MODEL_DIR = Path("/kaggle/working/mt5-shqip-LoRA") if IS_KAGGLE else PROJECT_ROOT / "models/mt5-shqip-LoRA"
PREDICTIONS_FILE = Path("/kaggle/working/test_predictions_with_summaries.csv") if IS_KAGGLE else PROJECT_ROOT / "data/test_predictions_with_summaries.csv"

set_seed(SEED)
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"Adapter output: {MODEL_DIR}")

## 2. Load and tokenize the dataset

The notebook first looks for the processed CSV files in `data/processed/`, then falls back to the Kaggle dataset path. Padding is performed dynamically by the data collator, so padding tokens in the labels are correctly replaced with `-100` and ignored by the loss function.

In [ ]:
DATA_DIR = PROJECT_ROOT / "data/processed"
KAGGLE_DATA_DIR = Path("/kaggle/input/datasets/heldilami/albaniansummarization")
if not (DATA_DIR / "train.csv").exists() and KAGGLE_DATA_DIR.exists():
    DATA_DIR = KAGGLE_DATA_DIR

data_files = {
    "train": DATA_DIR / "train.csv",
    "validation": DATA_DIR / "val.csv",
    "test": DATA_DIR / "test.csv",
}
missing_files = [str(path) for path in data_files.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing processed dataset files:\n" + "\n".join(missing_files)
    )

raw_datasets = load_dataset(
    "csv",
    data_files={name: str(path) for name, path in data_files.items()},
)
required_columns = {"source_text", "target_summary"}
missing_columns = required_columns - set(raw_datasets["train"].column_names)
if missing_columns:
    raise ValueError(f"Missing dataset columns: {sorted(missing_columns)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(batch):
    model_inputs = tokenizer(
        batch["source_text"],
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch["target_summary"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = raw_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
    desc="Tokenizing dataset",
)
test_df = raw_datasets["test"].to_pandas()
raw_datasets

## 3. Create the LoRA model

The base mT5 weights remain frozen. Only small LoRA matrices attached to the attention query and value projections are trained.

In [ ]:
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q", "v"],
    bias="none",
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

## 4. Configure training and evaluation

ROUGE-L on the validation set selects the best checkpoint. `bf16` is enabled only on supported GPUs. `fp16` remains disabled because it produced numerical instability with mT5 during the original experiments; unsupported GPUs safely fall back to `fp32`.

In [ ]:
rouge = evaluate.load("rouge")

def compute_metrics(eval_predictions):
    predictions, labels = eval_predictions
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predictions = np.where(
        (predictions >= 0) & (predictions < tokenizer.vocab_size),
        predictions,
        tokenizer.pad_token_id,
    )
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_predictions = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    scores = rouge.compute(
        predictions=decoded_predictions,
        references=decoded_labels,
        use_stemmer=False,
    )
    return {name: round(score * 100, 2) for name, score in scores.items()}

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
)
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

training_args = Seq2SeqTrainingArguments(
    output_dir=str(TRAINER_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    num_train_epochs=5,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=4,
    bf16=use_bf16,
    fp16=False,
    optim="adamw_torch",
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
print(f"Precision: {'bf16' if use_bf16 else 'fp32'}")

## 5. Train and save the adapter

Training evaluates and saves a checkpoint after every epoch, then restores the checkpoint with the best validation ROUGE-L. Only the compact LoRA adapter is exported; the base model and tokenizer remain available from `google/mt5-small`.

In [ ]:
train_result = trainer.train()
trainer.model.save_pretrained(MODEL_DIR, safe_serialization=True)
trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)
print(f"Best adapter saved to {MODEL_DIR}")

## 6. Evaluate on the held-out test set

The test set is used only after training and model selection are complete. `trainer.predict` performs batched generation, computes ROUGE, and returns the generated token IDs in one pass.

In [ ]:
test_output = trainer.predict(
    tokenized_datasets["test"],
    metric_key_prefix="test",
)
trainer.log_metrics("test", test_output.metrics)
trainer.save_metrics("test", test_output.metrics)

prediction_ids = test_output.predictions
if isinstance(prediction_ids, tuple):
    prediction_ids = prediction_ids[0]
prediction_ids = np.where(
    (prediction_ids >= 0) & (prediction_ids < tokenizer.vocab_size),
    prediction_ids,
    tokenizer.pad_token_id,
)
test_df["model_prediction"] = tokenizer.batch_decode(
    prediction_ids, skip_special_tokens=True
)
test_df[["target_summary", "model_prediction"]].head()

## 7. Compare against Lead-N baselines

Lead-N uses the first N words of each source article. The comparison uses exactly the same test references as the trained model.

In [ ]:
def lead_n(text, word_count):
    return " ".join(text.split()[:word_count])

references = test_df["target_summary"].tolist()
result_rows = [{
    "method": "mT5-small + LoRA",
    "ROUGE-1": test_output.metrics["test_rouge1"],
    "ROUGE-2": test_output.metrics["test_rouge2"],
    "ROUGE-L": test_output.metrics["test_rougeL"],
}]

for word_count in (8, 12, 20):
    column = f"lead_{word_count}"
    test_df[column] = test_df["source_text"].apply(lead_n, word_count=word_count)
    scores = rouge.compute(
        predictions=test_df[column].tolist(),
        references=references,
        use_stemmer=False,
    )
    result_rows.append({
        "method": f"Lead-{word_count}",
        "ROUGE-1": round(scores["rouge1"] * 100, 2),
        "ROUGE-2": round(scores["rouge2"] * 100, 2),
        "ROUGE-L": round(scores["rougeL"] * 100, 2),
    })

results_df = pd.DataFrame(result_rows).set_index("method")
results_df

## 8. Inspect qualitative examples

In [ ]:
sample = test_df.sample(10, random_state=SEED)
for _, row in sample.iterrows():
    print("=" * 80)
    print(f"SOURCE (start):\n{row['source_text'][:300]}...")
    print(f"\nREFERENCE:\n{row['target_summary']}")
    print(f"\nMODEL:\n{row['model_prediction']}\n")

## 9. Export predictions

The UTF-8 BOM keeps Albanian characters readable when the CSV is opened in Microsoft Excel.

In [ ]:
PREDICTIONS_FILE.parent.mkdir(parents=True, exist_ok=True)
test_df.to_csv(PREDICTIONS_FILE, index=False, encoding="utf-8-sig")
print(f"Predictions saved to {PREDICTIONS_FILE}")
results_df